<a href="https://colab.research.google.com/github/NaghamZidiah/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

FlyRank compared pages with rising impressions against pages with falling impressions. The growing group was longer on average (3.2K vs. 2.3K words), younger (184 vs. 230 days), and slightly better positioned in search (15.9 vs. 16.1 average position). The comparison used large groups of 74,187 rising pages and 45,272 falling pages.

The paper describes this as an observational comparison rather than causal evidence.

**My methodology question:** Because this is an observational comparison, how much of the difference between growing and declining pages could be explained by pre-existing differences between the two groups? For example, could topic, page type, or other page characteristics contribute to the observed differences in word count, age, and visibility?

This question does not challenge the observed differences themselves. It asks whether additional controls or a more comparable grouping would be needed before interpreting the differences as evidence that these characteristics contribute to growth.

### Finding 2 — The Content Performance Curve

FlyRank found that content performance peaks at 61–90 days, declines after 270 days, and that the rebound among pages older than 365 days is concentrated in pages that were refreshed.

**My methodology question:** Because the analysis is observational, were refreshed pages compared with a similar group of older pages that were not refreshed? For example, could differences in their baseline performance, content quality, topic, or other characteristics explain part of the observed rebound?

This question is important because a stronger comparison or matched control group could help distinguish an association between refreshing and improved performance from differences that already existed between the pages.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split

The ML-08 model used a random 80/20 split, which is convenient for reproducibility but does not reflect the temporal structure of performance data.

For this validation audit, I use a time-aware split. Earlier observations from March 2026 are used for training, while later observations are held out for testing.

This design better reflects a real prediction or decision-support setting because the model is evaluated on observations that occur after the training period.

I keep the same clustering features and evaluation metric so that the comparison focuses on the effect of the validation design rather than changing the modeling approach.

In [ ]:
# Load and prepare the data for the validation audit

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

cluster_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
"""

cluster_df = con.sql(cluster_query).df()

cluster_df = cluster_df.replace(
    [np.inf, -np.inf], np.nan
).dropna()

cluster_df["report_date"] = pd.to_datetime(
    cluster_df["report_date"]
)

print("Total rows:", len(cluster_df))
print("Earliest date:", cluster_df["report_date"].min())
print("Latest date:", cluster_df["report_date"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 3611061
Earliest date: 2026-03-01 00:00:00
Latest date: 2026-03-31 00:00:00


In [ ]:
# Create the same reproducible 100,000-row sample used in ML-08

model_sample = cluster_df.sample(
    n=min(100000, len(cluster_df)),
    random_state=42
).reset_index(drop=True)

# Create the transformed features used by the ML-08 model
model_sample["log_impressions"] = np.log1p(
    model_sample["gsc_impressions"]
)

model_sample["log_clicks"] = np.log1p(
    model_sample["gsc_clicks"]
)

print("Validation sample size:", len(model_sample))
print("Date range:",
      model_sample["report_date"].min(),
      "to",
      model_sample["report_date"].max())

display(model_sample.head())

Validation sample size: 100000
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,log_impressions,log_clicks
0,2026-03-25,client_23a62021009f63c4,content_04280acc2eed61c5,117,1,23.461538,4.770685,0.693147
1,2026-03-12,client_73cda7b4e4f265ea,content_94440f06057a720c,21,0,3.714286,3.091042,0.000000
2,2026-03-30,client_23a62021009f63c4,content_5e2c6ea8d269d39f,125,1,11.064000,4.836282,0.693147
3,2026-03-18,client_73cda7b4e4f265ea,content_6d1abdf4a75e8611,16,0,47.000000,2.833213,0.000000
4,2026-03-24,client_73cda7b4e4f265ea,content_a8df3264e941770b,35,0,9.657143,3.583519,0.000000


In [ ]:
# Create a time-aware train/test split

split_date = pd.Timestamp("2026-03-25")

train_cluster = model_sample[
    model_sample["report_date"] < split_date
].copy()

test_cluster = model_sample[
    model_sample["report_date"] >= split_date
].copy()

print("Split date:", split_date.date())

print("\nTraining period:")
print(train_cluster["report_date"].min(), "to",
      train_cluster["report_date"].max())

print("Training rows:", len(train_cluster))

print("\nTest period:")
print(test_cluster["report_date"].min(), "to",
      test_cluster["report_date"].max())

print("Test rows:", len(test_cluster))

Split date: 2026-03-25

Training period:
2026-03-01 00:00:00 to 2026-03-24 00:00:00
Training rows: 75639

Test period:
2026-03-25 00:00:00 to 2026-03-31 00:00:00
Test rows: 24361


In [ ]:
# Fit the ML-08 clustering model using a time-aware split

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Same features and k as the final ML-08 model
model_features = [
    "log_impressions",
    "log_clicks",
    "gsc_avg_position"
]

X_train = train_cluster[model_features].copy()
X_test = test_cluster[model_features].copy()

# Fit scaler only on training data
time_scaler = StandardScaler()

X_train_scaled = time_scaler.fit_transform(X_train)
X_test_scaled = time_scaler.transform(X_test)

# Same final model configuration as ML-08
time_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

# Fit only on training data
time_train_labels = time_model.fit_predict(X_train_scaled)

# Assign clusters to unseen test data
time_test_labels = time_model.predict(X_test_scaled)

# Evaluate separation on the test period
time_test_silhouette = silhouette_score(
    X_test_scaled,
    time_test_labels
)

print("Time-aware test silhouette score:",
      round(time_test_silhouette, 4))

print("\nTime-aware test cluster sizes:")
print(
    pd.Series(time_test_labels)
    .value_counts()
    .sort_index()
)

Time-aware test silhouette score: 0.5307

Time-aware test cluster sizes:
0    21766
1     2595
Name: count, dtype: int64


In [ ]:
# Compare random-split and time-aware validation

validation_comparison = pd.DataFrame({
    "validation_design": [
        "ML-08 random split",
        "ML-09 time-aware split"
    ],
    "test_silhouette": [
        0.5372,
        time_test_silhouette
    ]
})

validation_comparison["change"] = (
    validation_comparison["test_silhouette"]
    - validation_comparison.loc[0, "test_silhouette"]
)

display(validation_comparison.round(4))

,validation_design,test_silhouette,change
0,ML-08 random split,0.5372,0.0000
1,ML-09 time-aware split,0.5307,-0.0065


### Before vs. after validation design

The ML-08 model achieved a test silhouette score of 0.5372 using a random train-test split.

After applying a time-aware split, where the model was trained on March 1–24 and evaluated on March 25–31, the test silhouette score was 0.5307.

The score decreased slightly by 0.0065. This suggests that the clustering separation was broadly similar under the more realistic time-aware validation design, although the model did not perform exactly the same on later observations.

The time-aware result provides a more cautious validation of the model because later observations were kept separate from the training period.

This comparison does not show that the model is predictive of future performance. It only shows how the observed clustering structure changes when evaluated on a later time period.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The ML-08 clustering model uses three numerical features:

- `log_impressions`
- `log_clicks`
- `gsc_avg_position`

These features are derived from GSC performance metrics and were available within the observed reporting period.

The model does not use a target label because this is an unsupervised clustering task.

I also excluded identifiers such as `client_hash_id` and `content_hash_id` from the model features because they identify records rather than describe performance.

The model does not use future target-window metrics, predefined labels, `health_score`, `priority_score`, `action_type`, or other decision fields.

The time-aware split further reduces the risk of using information from later dates during model fitting because the scaler and K-Means model are fitted only on the training period.

However, these features describe observed performance during March 2026. Therefore, the model should be interpreted as describing performance patterns in the observed data, not as a leakage-free forecast of future content performance.

In [ ]:
# Check ML-08 model features for potential leakage indicators

model_features = [
    "log_impressions",
    "log_clicks",
    "gsc_avg_position"
]

risky_keywords = [
    "target",
    "label",
    "future",
    "trend",
    "decision",
    "score",
    "health",
    "action",
    "client_hash_id",
    "content_hash_id"
]

print("Model features:")
for feature in model_features:
    print("-", feature)

print("\nPotential leakage keyword check:")

for feature in model_features:
    matches = [
        keyword
        for keyword in risky_keywords
        if keyword in feature.lower()
    ]

    if matches:
        print(f"{feature}: REVIEW -> {matches}")
    else:
        print(f"{feature}: PASS")

Model features:
- log_impressions
- log_clicks
- gsc_avg_position

Potential leakage keyword check:
log_impressions: PASS
log_clicks: PASS
gsc_avg_position: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The final K-Means model separates content pages into lower-performance and higher-performance groups that can support content optimization decisions.

### Safer claim

In this March 2026 sample, the K-Means model **measured two broad performance-based groups** using impressions, clicks, and average position after log transformation of impressions and clicks. The observed cluster profiles show a larger group with lower average impressions and clicks and a smaller group with higher average impressions and clicks.

The model achieved a test silhouette score of **0.5372** under the ML-08 random split and **0.5307** under the ML-09 time-aware split. The small decrease under the time-aware validation provides a more cautious view of the observed cluster separation.

These results are **directional and decision-support oriented**. They describe performance patterns observed in this sample and should not be interpreted as proof that the model predicts future content performance or determines which pages should be optimized.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.